### Step 1: Initial Set Up and Load 
Loading in output from notebook 4 and maintaining train/test/val variables for comparison

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
                             precision_recall_curve, confusion_matrix)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

PROJECT_ROOT = Path.cwd().resolve()
FEATURES_DIR = PROJECT_ROOT / "data" / "features"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [2]:
case_features = pd.read_parquet(FEATURES_DIR / "case_features.parquet")
drug_matrix = sp.load_npz(FEATURES_DIR / "drug_multihot.npz").tocsr()
baseline_metrics = json.loads((FEATURES_DIR / "baseline_metrics.json").read_text())

RANDOM_STATE = 42
NUM_COLS   = ["age_years", "weight_kg"]
COUNT_COLS = ["n_drugs", "n_suspect_ingredients", "n_other_ingredients"]
FLAG_COLS  = ["age_missing", "weight_missing", "is_us"]
CAT_COLS   = ["sex", "age_group", "reporter", "report_type", "country"]


print(case_features.shape, drug_matrix.shape)

(8012423, 17) (8012423, 400)


### Step 2: X and Y setup
Reusing and reformatting train/test/val splits from Notebook 4 

In [ ]:
# extract train and val indices
train_idx = np.where(case_features["in_train_sample"] == True)[0]
val_idx = np.where(case_features["split"] == "val")[0]

#extract train and val dataframes from indices
train_df = case_features.iloc[train_idx]
val_df = case_features.iloc[val_idx]

# one-hot encode categorical columns
ohe = OneHotEncoder(handle_unknown="ignore")
ohe.fit(train_df[CAT_COLS].astype(str))
train_cat = ohe.transform(train_df[CAT_COLS].astype(str)).toarray().astype("float32")
val_cat = ohe.transform(val_df[CAT_COLS].astype(str)).toarray().astype("float32")

# extract numerical columns
train_num = train_df[NUM_COLS + COUNT_COLS + FLAG_COLS].to_numpy(dtype="float32")
val_num = val_df[NUM_COLS + COUNT_COLS + FLAG_COLS].to_numpy(dtype="float32")

# extract drug matrix
train_drugs = drug_matrix[train_idx].toarray().astype("float32")
val_drugs = drug_matrix[val_idx].toarray().astype("float32")

# combine features
X_train = np.hstack([train_num, train_cat, train_drugs])
X_val = np.hstack([val_num, val_cat, val_drugs])

y_train = train_df["y"].astype(int).to_numpy()
y_val = val_df["y"].astype(int).to_numpy()

print(X_train.shape, X_val.shape)
print(y_train.mean(), y_val.mean())

### Step 3: Run Baseline Gradient boosting model 

In [ ]:
# run a base gb without changing parameters 
gb = HistGradientBoostingClassifier(early_stopping=False, random_state=RANDOM_STATE)
gb.fit(X_train, y_train)

In [ ]:
# probabilities for the serious class
val_probs = gb.predict_proba(X_val)[:, 1]

# scores
gb_pr = average_precision_score(y_val, val_probs)
gb_roc = roc_auc_score(y_val, val_probs)

print("GB PR-AUC:", gb_pr, "| ROC-AUC:", gb_roc)

# compare with the logistic regression baseline from notebook 4
lr = baseline_metrics["logreg, all features"]
print("LogReg PR-AUC:", lr["val_avg_precision"], "| ROC-AUC:", lr["val_roc_auc"])

In [ ]:
test_idx = np.where(base_out["split"] == "test")[0]
test_df = base_out.iloc[test_idx]

test_cat = ohe.transform(test_df[CAT_COLS].astype(str)).toarray().astype("float32")
test_num = test_df[NUM_COLS + COUNT_COLS + FLAG_COLS].to_numpy(dtype="float32")
test_drugs = drug_matrix[test_idx].toarray().astype("float32")

X_test = np.hstack([test_num, test_cat, test_drugs])
y_test = test_df["y"].astype(int).to_numpy()

test_probs = gb.predict_proba(X_test)[:, 1]
print("TEST PR-AUC:", round(average_precision_score(y_test, test_probs), 4))
print("TEST ROC-AUC:", round(roc_auc_score(y_test, test_probs), 4))

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=73434ccc-8a09-4578-aac6-0389f491d707' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>